# TeCo: ZuCo-inspired exact-40 averaging and scalp regions

**CPU → Run all**, using **Zyion** for both Drive prompts. Do not change an active notebook's runtime.

**Primary question:** do a narrower window or fewer scalp-channel features help generalize to new readers?

- Same useful choices as the ZuCo 63.9% setup: positive/negative, **224–304 ms channel means**, fixed **C=1 balanced logistic regression with liblinear**, train-only imputation/scaling, three **reader-held-out** folds, and **exactly 40** same-class trials per reader/partition. Compare with individual trials. Same fold seed20261113; words/sentences may overlap across readers.
- **Spatial comparisons:** all TeCo channels, each of six existing scalp regions, six regional means, and a region chosen by **inner reader-grouped CV on training readers only**. All candidates use identical events/folds/groups. Fixed-region results are exploratory, not a license to report the best test region as validated.
- **Fair reference:** the original48regional0–800ms features are also tested with the same cohort and exact40groups. Pre-onset EEG, word/fixation-only and dummy controls run too. No regularization search or CNN sweep.
- Require≥40positive and≥40negative events per reader **before fitting**. Use the same qualifying cohort for all models and both sizes; show exclusions. Stop if fewer than6readers qualify. Discard incomplete groups. Five deterministic40-trial regroupings, one single-trial run. Show ranges across regroupings, **not confidence intervals**.
- **Remain different from paper-style ZuCo:** first stored fixation only, TeCo's own channel montage, original40Hz/125Hz epoch preprocessing, no extra regression cleaning or paper-style trial matching. Keep the frozen original TeCo trial labels. No relabeling, source downloads, onset recovery or new EEG extraction.

**Storage and resume:** at most **20 MB total new Drive outputs**, including small persistent channel-mean caches. The first run reads the existing saved TeCo epoch shards once per reader, takes224–304ms/pre-onset channel means and saves them. Later runs reuse verified means. No new full EEG copy. Completed model cases resume. Output: `Thesis/Results/word_sentiment_eeg/teco_spatial40/`.

Test labels construct the averages, so these are offline class-grouped predictions, not individual-word deployment accuracy. Results are descriptive and not an exact numerical replication of ZuCo.

The final section additionally audits historical ZuCo train/test **predictor-sample** overlap using metadata only. It reports counts only if reconstructed memberships match saved historical fold hashes. Missing/unverifiable historical data is explicitly flagged; it does not invalidate or stop the new TeCo experiment.


In [ ]:
# Runtime check: run before installation, Drive access, or analysis.
import os
import subprocess

REQUIRED_RUNTIME = 'CPU'
if os.environ.get('COLAB_TPU_ADDR') or os.environ.get('TPU_NAME'):
    raise RuntimeError(f'This notebook needs {REQUIRED_RUNTIME}, not TPU. Choose Runtime > Change runtime type, then rerun all.')
try:
    _gpu_probe = subprocess.run(
        ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
        capture_output=True, text=True, timeout=15, check=False)
except FileNotFoundError:
    _gpu_names = ''
except subprocess.TimeoutExpired:
    raise RuntimeError('GPU detection timed out. Reconnect the runtime and rerun this cell.') from None
else:
    if _gpu_probe.returncode:
        raise RuntimeError('GPU detection failed. Reconnect the runtime and rerun this cell.')
    _gpu_names = _gpu_probe.stdout.strip()
if REQUIRED_RUNTIME == 'GPU' and not _gpu_names:
    raise RuntimeError('GPU required. Choose Runtime > Change runtime type > T4 GPU, then Run all. No setup or analysis has started.')
if REQUIRED_RUNTIME == 'CPU' and _gpu_names:
    raise RuntimeError('CPU notebook: choose Runtime > Change runtime type > CPU (or hardware accelerator None), then Run all to avoid using GPU quota.')
print('Runtime verified:', _gpu_names if _gpu_names else 'CPU', flush=True)


## 1. Connect and check the source and storage


In [ ]:
from google.colab import drive, auth
from pathlib import Path
import os, sys, subprocess, json
from IPython.display import display, Markdown, Image
import pandas as pd

drive.mount('/content/drive')
THESIS=Path('/content/drive/MyDrive/Thesis')
OUT=THESIS/'Results/word_sentiment_eeg/teco_spatial40'
REPO=Path('/content/word-sentiment-eeg-teco-spatial40')
CODE_REVISION='2b26cbe256f3354dc2cf083905e812d1920c8852'
def git(*args):
    result=subprocess.run(['git','-c','credential.helper=','-c','http.extraheader=',*map(str,args)],
                          text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                          env=dict(os.environ,GIT_TERMINAL_PROMPT='0'))
    if result.returncode:
        raise RuntimeError('Git setup failed:\n'+result.stdout)
    if result.stdout:print(result.stdout,end='',flush=True)
if not REPO.exists():
    git('clone','--depth','1','https://github.com/parmisbathayan/word-sentiment-eeg.git',REPO)
else:
    if subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True).strip():
        raise RuntimeError('Temporary checkout contains edits. Preserve them before updating.')
if subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()!=CODE_REVISION:
    git('-C',REPO,'fetch','--depth','1','origin',CODE_REVISION)
    fetched=subprocess.check_output(['git','-C',str(REPO),'rev-parse','FETCH_HEAD^{commit}'],text=True).strip()
    if fetched!=CODE_REVISION:raise RuntimeError('Fetched code differs from the pinned commit.')
    git('-C',REPO,'checkout','--detach','FETCH_HEAD')
if subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()!=CODE_REVISION:
    raise RuntimeError('Code checkout verification failed.')
# Temporary cloud-only scientific environment: no base-runtime upgrades/restart.
# BOOTSTRAP_START: also exercised verbatim in cloud CI.
# Use a file-rendering backend in subprocesses, not Colab's unavailable inline plugin.
os.environ['MPLBACKEND']='Agg'
ENV=Path('/content/word-eeg-paper-classification-python')
PYTHON=ENV/'bin/python'
def setup_command(command):
    print('SETUP:', ' '.join(map(str,command)), flush=True)
    result=subprocess.run(list(map(str,command)),stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
    if result.stdout:print(result.stdout,flush=True)
    if result.returncode:
        raise RuntimeError('Scientific environment setup failed; the full diagnostic is printed above. No EEG processing started.')
# Colab may lack ensurepip. This also repairs a partially created environment.
setup_command([sys.executable,'-m','venv','--without-pip',str(ENV)])
setup_command([sys.executable,'-m','pip','--python',str(PYTHON),'install',
               '--only-binary=:all:','-r',str(REPO/'requirements-paper-classification.txt')])
setup_command([sys.executable,'-m','pip','--python',str(PYTHON),'check'])
setup_command([str(PYTHON),'-c',
    'import numpy,pandas,scipy,sklearn,matplotlib; '
    'print("Scientific environment ready:",numpy.__version__,pandas.__version__,scipy.__version__,sklearn.__version__,matplotlib.__version__)'])
# BOOTSTRAP_END
print('Code revision:',subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip())

# Verify account quota before loading any EEG. Use Zyion for both prompts.
auth.authenticate_user()
import google.auth
from google.auth.transport.requests import AuthorizedSession
credentials,_=google.auth.default()
response=AuthorizedSession(credentials).get('https://www.googleapis.com/drive/v3/about',
    params={'fields':'storageQuota,user(emailAddress)'},timeout=30)
response.raise_for_status();about=response.json();quota=about['storageQuota']
email=about.get('user',{}).get('emailAddress','unavailable')
print('Authenticated quota account:',email)
if email!='zyionygvmutch39@gmail.com':
    raise RuntimeError('Choose your Zyion account for Drive authentication and mounting; it contains the saved trials.')
if 'limit' not in quota:
    raise RuntimeError('Drive did not expose a storage limit; cannot verify free space.')
FREE_DRIVE_BYTES=max(0,int(quota['limit'])-int(quota.get('usage',0)))
used=sum(p.stat().st_size for p in OUT.rglob('*') if p.is_file()) if OUT.exists() else 0
if FREE_DRIVE_BYTES<max(0,20_000_000-used):
    raise RuntimeError('Reserve at most 20 MB for results. This notebook reads only saved TeCo epochs once for channel means; no new extraction.')
print(f'Free space: {FREE_DRIVE_BYTES/1e9:.2f} GB; new output cap: 20 MB.')
print('New output folder:',OUT)


## 2. Prepare small channel means and fit comparisons
Saved-reader progress is printed. Whole-channel means persist; there is no need to repeat their preparation after a disconnect.


In [ ]:
sys.path.insert(0,str(REPO/'scripts'))
from colab_progress import run_with_progress
command=[str(PYTHON),'-u',str(REPO/'scripts/teco_spatial40.py'),
         '--thesis-root',str(THESIS),'--free-drive-bytes',str(FREE_DRIVE_BYTES)]
run_with_progress(command,OUT/'runtime_status.json',activity='TeCo exact40 channel/region comparison: saved epochs to reusable small means')


## 3. Cohort, scores and actual test counts


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
status=json.loads((OUT/'complete.json').read_text())
if status['status']!='completed':raise RuntimeError('Experiment is incomplete')
import hashlib
for relative,sha in status['checksums'].items():
    path=(OUT/relative).resolve()
    if not path.is_relative_to(OUT.resolve()) or hashlib.sha256(path.read_bytes()).hexdigest()!=sha:
        raise ValueError('Saved result integrity check failed: '+relative)
scores=pd.read_csv(OUT/'scores.csv')
summary=scores.groupby(['variant','size']).agg(mean_ba=('balanced_accuracy','mean'),minimum=('balanced_accuracy','min'),
    maximum=('balanced_accuracy','max'),regroupings=('regrouping','size'),test_examples=('test_examples','first'),
    original_test_trials=('original_test_trials','first'),readers=('readers','first')).reset_index()
display(pd.read_csv(OUT/'reader_cohort.csv'))
view=summary.copy()
for c in ['mean_ba','minimum','maximum']:view[c]=(100*view[c]).round(1)
display(view)
print('Primary: whole_head versus selected_region. All fixed region scores remain exploratory.')


## 4. Primary comparison and controls


In [ ]:
names=['whole_head','selected_region','six_region_means','cached_0_800','pre_control','word_eye_control','dummy']
labels=['All channels, 224–304 ms','Training-selected region, 224–304 ms','Six regional means, 224–304 ms','Original regional bins, 0–800 ms','Pre-onset EEG control','Word/fixation properties only','Dummy']
fig,ax=plt.subplots(figsize=(10,6))
for size,offset,color in [(1,-.12,'#2874a6'),(40,.12,'#148f77')]:
    part=summary[summary['size'].eq(size)].set_index('variant').loc[names]
    mean=100*part.mean_ba.to_numpy();low=mean-100*part.minimum.to_numpy();high=100*part.maximum.to_numpy()-mean
    y=np.arange(len(names))+offset
    ax.errorbar(mean,y,xerr=np.maximum(0,np.stack([low,high])),fmt='o',capsize=3,color=color,label='Individual trials' if size==1 else 'Exact40; mean and grouping range')
    for x,yy in zip(mean,y):ax.annotate(f'{x:.1f}%',(x,yy),xytext=(5,0),textcoords='offset points',va='center',fontsize=9)
ax.axvline(50,color='black',ls=':',lw=1);ax.set(yticks=np.arange(len(names)),yticklabels=labels,xlim=(0,105),xlabel='Balanced accuracy (%)',title='TeCo task1: unseen readers, same cohort and folds')
ax.invert_yaxis();ax.legend(loc='lower right',fontsize=8)
fig.tight_layout();display(fig);plt.close(fig)
print('Ranges reflect five regroupings, not significance or confidence intervals. All test averages contain exactly40 trials.')


## 5. Fixed regions


In [ ]:
regions=['anterior','central','posterior','occipital','left_lateral','right_lateral']
fig,ax=plt.subplots(figsize=(10,5))
for size,offset,color in [(1,-.12,'#2874a6'),(40,.12,'#148f77')]:
    part=summary[summary['size'].eq(size)].set_index('variant').loc[['region_'+r for r in regions]]
    mean=100*part.mean_ba.to_numpy()
    ax.errorbar(mean,np.arange(6)+offset,xerr=np.maximum(0,np.stack([mean-100*part.minimum.to_numpy(),100*part.maximum.to_numpy()-mean])),fmt='o',capsize=3,color=color,label=f'{size} trial(s)/example')
ax.axvline(50,color='black',ls=':',lw=1);ax.set(yticks=np.arange(6),yticklabels=regions,xlim=(0,100),xlabel='Balanced accuracy (%)',title='Predefined scalp regions, 224–304 ms; exploratory comparisons')
ax.invert_yaxis();ax.legend();fig.tight_layout();display(fig);plt.close(fig)
print('These labels describe scalp electrode groups, not isolated brain sources. Do not choose the best region using these outer-test scores.')


## 6. Reader consistency and training-only region choices


In [ ]:
reader_rows=[];chosen=[]
for path in sorted(OUT.glob('size40/repeat*/*/result.json')):
    saved=json.loads(path.read_text());variant=saved['score']['variant']
    if variant in ['whole_head','selected_region','cached_0_800']:
        reader_rows.extend(dict(variant=variant,regrouping=saved['score']['regrouping'],**r) for r in saved['per_reader'])
    if variant=='selected_region':chosen.extend(dict(regrouping=saved['score']['regrouping'],**r) for r in saved['selected_regions'])
r=pd.DataFrame(reader_rows)
display((100*r.groupby(['reader','variant']).balanced_accuracy.mean().unstack()).round(1))
display(pd.DataFrame(chosen)[['regrouping','fold','chosen']])
print('Each region above was selected using training readers only. No test reader helped choose its own region.')


## 7. Historical ZuCo overlap audit
This is metadata-only, separate from the new TeCo experiment. Counts are withheld when original event memberships cannot be verified.


In [ ]:
audit=json.loads((OUT/'zuco_overlap_audit.json').read_text())
verified=pd.DataFrame(audit['verified_rows'])
if len(verified):
    counts=['train_events','test_events','overlapping_train_events','overlapping_test_events','cross_partition_event_pairs']
    display(verified.groupby(['cohort','split','window','size'])[counts].sum().reset_index())
else:print('No historical overlap counts could be verified. See reasons below; do not assume zero overlap.')
if audit['unverified']:display(pd.DataFrame(audit['unverified']))
print('Counts concern direct reuse of predictor samples, not just overlap of the stored epochs.')
print('A 224–304ms feature spans80ms; the broader0–800ms features can overlap more nearby fixation inputs.')
print('The audit does not assess the influence of filtering or full-reader regression cleaning.')
